In [1]:
import math
from typing import Generator

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
from tqdm import tqdm

from zoo.utils import get_device

In [2]:
dtype = torch.float16
device = get_device()
print(f"Using dtype: {dtype}")
print(f"Using device: {device}")


Using dtype: torch.float16
Using device: mps


In [3]:
class Tokenizer:
    def __init__(self, dataset: str):
        self.vocab = sorted(set(dataset))
        self.word2idx = {ch: idx for idx, ch in enumerate(self.vocab)}
        self.idx2word = {idx: ch  for idx, ch in enumerate(self.vocab)}

    def __len__(self):
        return len(self.vocab)

    def encode(self, text: str) -> list[int]:
        return [self.word2idx[ch] for ch in text]

    def decode(self, indices: list[int]) -> str:
        return "".join(self.idx2word[idx] for idx in indices)


with open("../data/tinyshakespeare.txt", "r") as f:
    dataset = f.read()

tokenizer = Tokenizer(dataset)
vocab_size = len(tokenizer)

split = int(len(dataset) * 0.8)

# Keep as flat 1-D tensors on CPU — batches() will slice windows
train_flat = torch.tensor(tokenizer.encode(dataset[:split]), dtype=torch.long)
test_flat  = torch.tensor(tokenizer.encode(dataset[split:]),  dtype=torch.long)

block_size = 128


def batches(
    data: torch.Tensor,
    batch_size: int,
) -> Generator[tuple[torch.Tensor, torch.Tensor], None, None]:
    # data is a flat 1-D token tensor.
    # Each sample: x = data[i : i+block_size], y = data[i+1 : i+block_size+1]
    # i.e. y is x shifted right by one token — standard causal LM target.
    max_start = len(data) - block_size - 1
    starts = torch.randperm(max_start)  # shuffle every epoch
    for i in range(0, len(starts) - batch_size, batch_size):
        idx = starts[i : i + batch_size]          # (B,)
        x = torch.stack([data[j : j + block_size]     for j in idx])  # (B, T)
        y = torch.stack([data[j+1 : j + block_size+1] for j in idx])  # (B, T)
        yield x, y


print(f"Vocab size:    {vocab_size}")
print(f"Train tokens:  {len(train_flat):,}")
print(f"Test tokens:   {len(test_flat):,}")
print(f"Steps/epoch:   {(len(train_flat) - block_size - 1) // 128}")
print(f"Sample: {tokenizer.decode(train_flat[:block_size].tolist())!r}")


Vocab size:    65
Train tokens:  892,314
Test tokens:   223,079
Steps/epoch:   6970
Sample: 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to '


In [4]:
import math
from typing import Generator

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary


# ── RoPE ──────────────────────────────────────────────────────────────────────
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_length=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_length = max_seq_length
        self._build_cache(max_seq_length)

    def _build_cache(self, seq_len):
        t    = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)
        emb   = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None], persistent=False)

    def forward(self, seq_len: int):
        # Rebuild cache if device moved after init (e.g. .to(mps) after construction)
        if self.cos_cached.device != self.inv_freq.device:
            self._build_cache(self.max_seq_length)
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    return (q * cos + rotate_half(q) * sin,
            k * cos + rotate_half(k) * sin)


# ── Attention ─────────────────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads, max_seq_length=2048):
        super().__init__()
        assert embed_size % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_size // num_heads
        self.qkv    = nn.Linear(embed_size, 3 * embed_size, bias=False)
        self.fc_out = nn.Linear(embed_size, embed_size, bias=False)
        self.rope   = RotaryPositionalEmbedding(self.head_dim, max_seq_length)

    def forward(self, x):
        B, S, E = x.shape
        Q, K, V = self.qkv(x).view(B, S, 3, self.num_heads, self.head_dim).unbind(2)
        Q, K, V = Q.transpose(1, 2), K.transpose(1, 2), V.transpose(1, 2)
        cos, sin = self.rope(S)
        Q, K = apply_rope(Q, K, cos, sin)
        # Explicit causal mask avoids Inductor symbolic-shape bug on MPS
        mask = torch.ones(S, S, dtype=torch.bool, device=x.device).tril()
        out  = F.scaled_dot_product_attention(Q, K, V, attn_mask=mask)
        return self.fc_out(out.transpose(1, 2).contiguous().view(B, S, E))


# ── Norms ─────────────────────────────────────────────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, embed_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(embed_size))
        self.eps = eps

    def forward(self, x):
        # Cast to float32 for the norm computation — critical for fp16 stability.
        # rsqrt of a near-zero mean^2 in fp16 easily produces inf/NaN.
        x_f32 = x.float()
        norm   = x_f32 * torch.rsqrt(x_f32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(x.dtype)


# ── MLP (SwiGLU) ──────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, embed_size, expansion_factor=4):
        super().__init__()
        hidden = int(embed_size * expansion_factor * 2 / 3)
        hidden = (hidden + 7) // 8 * 8
        self.gate = nn.Linear(embed_size, hidden, bias=False)
        self.up   = nn.Linear(embed_size, hidden, bias=False)
        self.down = nn.Linear(hidden, embed_size, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


# ── MoE ───────────────────────────────────────────────────────────────────────
class MixtureOfExperts(nn.Module):
    def __init__(self, embed_size, num_experts=4, top_k=2, expansion_factor=4):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        hidden = int(embed_size * expansion_factor * 2 / 3)
        hidden = (hidden + 7) // 8 * 8

        # Init in float32 regardless of model dtype — moved to device dtype by .to()
        self.w_gate = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_up   = nn.Parameter(torch.empty(num_experts, embed_size, hidden))
        self.w_down = nn.Parameter(torch.empty(num_experts, hidden, embed_size))
        self.router = nn.Linear(embed_size, num_experts, bias=False)

        for w in (self.w_gate, self.w_up, self.w_down):
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))

    def forward(self, x):
        B, S, E = x.shape
        T = B * S
        x_flat = x.view(T, E)

        logits   = self.router(x_flat)
        topk_w, topk_ids = torch.topk(logits, self.top_k, dim=-1)
        topk_w   = F.softmax(topk_w, dim=-1)

        gate = torch.zeros(T, self.num_experts, device=x.device, dtype=x.dtype)
        gate.scatter_(1, topk_ids, topk_w)

        h_gate = torch.einsum("te,neh->tnh", x_flat, self.w_gate)
        h_up   = torch.einsum("te,neh->tnh", x_flat, self.w_up)
        h = F.silu(h_gate) * h_up
        h = h * gate.unsqueeze(-1)
        out = torch.einsum("tnh,nhe->te", h, self.w_down)

        return out.view(B, S, E)


# ── Block ─────────────────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, embed_size, num_heads, block_num):
        super().__init__()
        self.attn  = MultiHeadAttention(embed_size, num_heads)
        self.ffn   = MixtureOfExperts(embed_size)
        self.norm1 = RMSNorm(embed_size)
        self.norm2 = RMSNorm(embed_size)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


# ── LLM ───────────────────────────────────────────────────────────────────────
class LLM(nn.Module):
    def __init__(self, vocab_size, embed_size, num_blocks, num_heads=4):
        super().__init__()
        self.embedding    = nn.Embedding(vocab_size, embed_size)
        self.blocks       = nn.ModuleList([Block(embed_size, num_heads, i) for i in range(num_blocks)])
        self.norm_out     = RMSNorm(embed_size)
        self.output_layer = nn.Linear(embed_size, vocab_size, bias=False)
        # Weight tying: safe because both stay in the same dtype after .to()
        self.output_layer.weight = self.embedding.weight

        # Initialise embedding with small std — prevents fp16 overflow at start
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.norm_out(x))


# Build in float32, move to device — autocast handles fp16 per-op
model = LLM(vocab_size, embed_size=64, num_blocks=8).to(device)
model = torch.compile(model, backend="aot_eager")

summary(model, input_size=(1, block_size), dtypes=[torch.long], device=device)


W0304 01:19:04.229000 97160 torch/_dynamo/convert_frame.py:1016] [4/8] torch._dynamo hit config.recompile_limit (8)
W0304 01:19:04.229000 97160 torch/_dynamo/convert_frame.py:1016] [4/8]    function: 'hook' (/Users/karan/Code/projects/zoo/.venv/lib/python3.13/site-packages/torchinfo/torchinfo.py:592)
W0304 01:19:04.229000 97160 torch/_dynamo/convert_frame.py:1016] [4/8]    last reason: 4/7: global_layer_info[4636013440].macs == 0                
W0304 01:19:04.229000 97160 torch/_dynamo/convert_frame.py:1016] [4/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0304 01:19:04.229000 97160 torch/_dynamo/convert_frame.py:1016] [4/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.


Layer (type:depth-idx)                                       Output Shape              Param #
OptimizedModule                                              [1, 128, 65]              --
├─LLM: 1-1                                                   [1, 128, 65]              --
│    └─Embedding: 2-1                                        [1, 128, 64]              4,160
│    └─ModuleList: 2-2                                       --                        --
│    │    └─Block: 3-1                                       [1, 128, 64]              151,936
│    │    └─Block: 3-2                                       [1, 128, 64]              151,936
│    │    └─Block: 3-3                                       [1, 128, 64]              151,936
│    │    └─Block: 3-4                                       [1, 128, 64]              151,936
│    │    └─Block: 3-5                                       [1, 128, 64]              151,936
│    │    └─Block: 3-6                                       [1, 12

In [9]:
import time


@torch.inference_mode()
def generate(model, tokenizer, prompt, max_length=128, temperature=1.0, verbose=True):
    was_training = model.training
    model.eval()

    input_ids = torch.tensor(
        tokenizer.encode(prompt), dtype=torch.long, device=device
    ).unsqueeze(0)

    start_time = time.time()
    for _ in range(max_length - len(input_ids[0])):
        with torch.autocast(device_type=str(device), dtype=torch.float16):
            output = model(input_ids)
        next_token_logits = output[:, -1, :].float() / temperature  # fp32 for sampling
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)
    end_time = time.time()

    if was_training:
        model.train()  # restore mode so training resumes correctly

    decoded = tokenizer.decode(input_ids.squeeze().tolist())
    if verbose:
        n_new = input_ids.size(1) - len(tokenizer.encode(prompt))
        print(f"Total Time: {end_time - start_time:.4f}s  |  "
              f"{n_new / (end_time - start_time):.1f} tok/s")
    return decoded


print(generate(model, tokenizer, max_length=64, prompt="MIRANDA:"))


Total Time: 1.6490s  |  34.0 tok/s
MIRANDA:K:ssP grjoFasb usatt trKAahanydpeBltek mYotr h ?esd srrd


In [6]:
steps_per_epoch = (len(train_flat) - block_size - 1) // 128
total_steps     = steps_per_epoch * 10  # 10 epochs

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

# Cosine decay with warmup — far better than LinearLR for language models
def lr_lambda(step):
    warmup = 100
    if step < warmup:
        return step / warmup
    progress = (step - warmup) / max(1, total_steps - warmup)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))  # decay to 10% of peak

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
print(f"Steps per epoch: {steps_per_epoch}  |  Total steps: {total_steps}")


Steps per epoch: 6970  |  Total steps: 69700


In [7]:
for epoch in range(10):
    model.train()
    pbar = tqdm(
        batches(train_flat, batch_size=128),
        total=(len(train_flat) - block_size - 1) // 128,
        desc=f"Epoch {epoch + 1}",
    )

    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # device_type must be a string ("mps"), not a torch.device object
        with torch.autocast(device_type=str(device), dtype=torch.float16):
            logits = model(x)                             # (B, T, vocab)
            loss   = criterion(logits.view(-1, vocab_size), y.view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # .item() on a fp16 scalar can return inf if loss overflowed — cast first
        loss_val = loss.float().item()
        pbar.set_postfix(loss=f"{loss_val:.4f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")

    print(generate(model, tokenizer, max_length=128, prompt="MIRANDA:"))


Epoch 1:   3%|▎         | 230/6970 [01:47<52:17,  2.15it/s, loss=2.8472, lr=3.0e-04]


KeyboardInterrupt: 

In [8]:
gradients = []
for name, param in model.named_parameters():
    if param.grad is not None:
        weight_norm = param.data.norm().item()
        grad_norm = param.grad.norm().item()
        ratio = grad_norm / (param.data.norm().item() + 1e-8)  # Avoid division by zero
        gradients.append(
            {
                "name": name,
                "weight_norm": weight_norm,
                "grad_norm": grad_norm,
                "ratio": ratio,
            }
        )

# Show all
show_weights = False
show_biases = True

print(f"{'Name':<45} {'Weight Norm':<25} {'Grad Norm':<25} {'Ratio':<25}")
for grad in gradients:
    if (show_weights and "weight" in grad["name"]) or (
        show_biases and "bias" in grad["name"]
    ):
        print(
            f"{grad['name']:<45} {grad['weight_norm']:<25.4e} {grad['grad_norm']:<25.4e} {grad['ratio']:<25.4e}"
        )

Name                                          Weight Norm               Grad Norm                 Ratio                    
